# Sistema de Registro de Partidos
## ¿Qué vamos a construir?

A lo largo de este notebook vamos a construir **paso a paso** un sistema
que permite:

1. Registrar resultados de partidos de fútbol en un archivo de texto.
2. Leer y analizar esos resultados.
3. Manejar errores sin que el programa explote.
4. Guardar estadísticas de equipos en formato JSON para que persistan entre sesiones.

Al final del notebook tendrás algo que se parece a lo que usan
aplicaciones reales de estadísticas deportivas.

---
>  **Cómo usar este notebook:** ejecuta cada celda con `Shift + Enter`,
> lee el contexto, modifica los ejemplos y observa qué cambia.


---

## Sección 1 — Escribir y leer archivos de texto

### El problema

Imagina que eres analista de datos de una liga de fútbol amateur.
Después de cada jornada, recibes los resultados en texto plano,
uno por línea, con este formato:

```
Colombia 2-1 Argentina
Brasil 0-0 Uruguay
Ecuador 3-2 Peru
```


### Leer el archivo completo

Ahora leemos el archivo de un golpe con `read_text()`.
Esto funciona bien porque es un archivo pequeño.


In [25]:
from pathlib import Path
path_j1 = Path("jornada_1.txt")

contenido = path_j1.read_text(encoding="utf-8")
print(contenido)

Colombia 2-1 Argentina
Brasil 0-0 Uruguay
Ecuador 3-2 Peru
Chile 1-4 Venezuela
Bolivia 0-2 Paraguay



### Leer línea por línea: analizar cada partido

`read_text()` carga todo en memoria. Para archivos grandes
(imagina una temporada entera con miles de partidos) es mejor
usar **streaming**: leer una línea a la vez sin cargar todo.

Compara las dos formas:


In [26]:
print("── Forma 1: read_text() + splitlines() ─────────────────")
print("   Carga TODO el archivo en RAM y luego lo divide.")
contenido = path_j1.read_text(encoding="utf-8")
for linea in contenido.splitlines():
    print(" ", linea)

print()
print("── Forma 2: with open() + loop ──────────────────────────")
print("   Lee UNA línea a la vez. Mucho más eficiente en archivos grandes.")
with path_j1.open(encoding="utf-8") as f:
    for linea in f:
        print(" ", linea.strip())

── Forma 1: read_text() + splitlines() ─────────────────
   Carga TODO el archivo en RAM y luego lo divide.
  Colombia 2-1 Argentina
  Brasil 0-0 Uruguay
  Ecuador 3-2 Peru
  Chile 1-4 Venezuela
  Bolivia 0-2 Paraguay

── Forma 2: with open() + loop ──────────────────────────
   Lee UNA línea a la vez. Mucho más eficiente en archivos grandes.
  Colombia 2-1 Argentina
  Brasil 0-0 Uruguay
  Ecuador 3-2 Peru
  Chile 1-4 Venezuela
  Bolivia 0-2 Paraguay


> **Reflexión:** para este archivo de 5 líneas da igual.
> Pero si tuvieras los resultados de todas las ligas del mundo
> durante 20 años, la diferencia sería enorme.
> **La buena costumbre se construye desde el principio.**

### Extraer datos de cada línea

Cada línea tiene el formato `Local GOLES-GOLES Visitante`.
Vamos a parsearlo:


In [27]:
def parsear_partido(linea):
    """
    Recibe: 'Colombia 2-1 Argentina'
    Devuelve: dict con local, visitante, goles_local, goles_visitante
    """
    partes = linea.strip().split()
    # partes = ['Colombia', '2-1', 'Argentina']
    local      = partes[0]
    visitante  = partes[2]
    goles      = partes[1].split("-")
    goles_local     = int(goles[0])
    goles_visitante = int(goles[1])

    return {
        "local": local,
        "visitante": visitante,
        "goles_local": goles_local,
        "goles_visitante": goles_visitante
    }

partido = parsear_partido("Colombia 2-1 Argentina")
print(partido)

{'local': 'Colombia', 'visitante': 'Argentina', 'goles_local': 2, 'goles_visitante': 1}


In [28]:

print(f"{'PARTIDO':<30} {'RESULTADO':<12} GANADOR")
print("─" * 55)

with path_j1.open(encoding="utf-8") as f:
    for linea in f:
        if not linea.strip():
            continue
        p = parsear_partido(linea)
        if p["goles_local"] > p["goles_visitante"]:
            ganador = p["local"]
        elif p["goles_local"] < p["goles_visitante"]:
            ganador = p["visitante"]
        else:
            ganador = "Empate"

        partido_str = f"{p['local']} vs {p['visitante']}"
        resultado   = f"{p['goles_local']}-{p['goles_visitante']}"
        print(f"{partido_str:<30} {resultado:<12} {ganador}")

PARTIDO                        RESULTADO    GANADOR
───────────────────────────────────────────────────────
Colombia vs Argentina          2-1          Colombia
Brasil vs Uruguay              0-0          Empate
Ecuador vs Peru                3-2          Ecuador
Chile vs Venezuela             1-4          Venezuela
Bolivia vs Paraguay            0-2          Paraguay


### Agregar más jornadas con modo `'a'` (append)

El modo `'w'` **sobrescribiría** el archivo. Queremos **agregar**
la jornada 2 sin borrar la 1. Para eso usamos `'a'`.


In [29]:
resultados_jornada2 = [
    "Argentina 3-0 Bolivia",
    "Uruguay 1-1 Colombia",
    "Venezuela 2-1 Brasil",
]

path_historico = Path("historico.txt")
path_historico.write_text(contenido, encoding="utf-8")

# Agregamos jornada 2 SIN borrar la 1
with path_historico.open("a", encoding="utf-8") as f:
    for resultado in resultados_jornada2:
        f.write(resultado + "\n")

print("Histórico completo:")
print(path_historico.read_text(encoding="utf-8"))

Histórico completo:
Colombia 2-1 Argentina
Brasil 0-0 Uruguay
Ecuador 3-2 Peru
Chile 1-4 Venezuela
Bolivia 0-2 Paraguay
Argentina 3-0 Bolivia
Uruguay 1-1 Colombia
Venezuela 2-1 Brasil



> **Trampa clásica:** si usaras `write_text()` o modo `'w'`
> en lugar de `'a'`, perderías la jornada 1. Sin warning, sin undo.


---

## Sección 2 — Excepciones: cuando los datos mienten

### El problema

Los datos del mundo real nunca son perfectos.
Imagina que alguien te manda los resultados así:

```
Colombia 2-1 Argentina   ← OK
Brasil - Uruguay          ← ¿Qué pasa acá? Faltan los goles
Ecuador 3-x Peru          ← 'x' no es un número
                          ← Línea vacía
Chile 1-4 Venezuela       ← OK
```

Si tu programa no maneja estos casos, **explota** con el primer error
y pierdes todos los demás datos. Vamos a blindarlo.


In [30]:
# Primero veamos QUÉ tipo de error lanza nuestro parser
# con datos corruptos

# Caso 1: formato incorrecto
# parsear_partido("Brasil - Uruguay")

# Caso 2: gol no numérico
# parsear_partido("Ecuador 3-x Peru")

# Caso 3: línea vacía
# parsear_partido("")

### `try-except`: estructura completa

```
try:       → intenta ejecutar
except:    → captura el error
else:      → si NO hubo error
finally:   → siempre corre (haya error o no)
```

La parte que más se omite en los libros: **capturar el error como variable**
con `as e` para poder inspeccionarlo:


In [31]:
def parsear_partido_seguro(linea):
    """
    Versión robusta del parser. Devuelve el dict si OK, None si hay error.
    """
    try:
        partes = linea.strip().split()
        if len(partes) != 3:
            raise ValueError(f"Formato inválido: se esperaban 3 partes, llegaron {len(partes)}")

        local     = partes[0]
        visitante = partes[2]
        goles     = partes[1].split("-")

        if len(goles) != 2:
            raise ValueError(f"El marcador '{partes[1]}' no tiene formato GOLES-GOLES")

        goles_local     = int(goles[0])   # lanza ValueError si no es número
        goles_visitante = int(goles[1])

    except ValueError as e:
        print(f"Línea ignorada — {e}")
        return None
    else:
        # Solo llegamos acá si NO hubo error
        return {
            "local": local,
            "visitante": visitante,
            "goles_local": goles_local,
            "goles_visitante": goles_visitante
        }

# Probamos con datos buenos y malos mezclados
lineas_prueba = [
    "Colombia 2-1 Argentina",    # OK
    "Brasil - Uruguay",          # formato roto
    "Ecuador 3-x Peru",          # gol no numérico
    "",                          # vacío
    "Chile 1-4 Venezuela",       # OK
]

print("Procesando líneas:")
for linea in lineas_prueba:
    resultado = parsear_partido_seguro(linea)
    if resultado:
        print(f"{resultado['local']} {resultado['goles_local']}-{resultado['goles_visitante']} {resultado['visitante']}")

Procesando líneas:
Colombia 2-1 Argentina
Línea ignorada — invalid literal for int() with base 10: ''
Línea ignorada — invalid literal for int() with base 10: 'x'
Línea ignorada — Formato inválido: se esperaban 3 partes, llegaron 0
Chile 1-4 Venezuela


### El error que comete todo el mundo: `except:` desnudo

Mira la diferencia entre estas tres formas:


In [32]:
# MAL: captura TODO, incluso Ctrl+C y errores del sistema
# try:
#     parsear_partido("algo")
# except:
#     print("error")

# ACEPTABLE: captura errores de la aplicación pero deja pasar Ctrl+C
# try:
#     parsear_partido("algo")
# except Exception:
#     print("error")

# IDEAL: captura exactamente lo que esperamos
try:
    parsear_partido("algo roto aqui")
except (ValueError, IndexError) as e:
    print(f"Error capturado: {type(e).__name__}: {e}")

print("El programa sigue vivo")

Error capturado: ValueError: invalid literal for int() with base 10: 'roto'
El programa sigue vivo


### Manejar archivo que no existe

¿Qué pasa si alguien pide una jornada que no se jugó?


In [33]:
def leer_jornada(numero):
    """Lee los resultados de una jornada. Maneja el caso de que no exista."""
    path = Path(f"jornada_{numero}.txt")
    try:
        contenido = path.read_text(encoding="utf-8")
    except FileNotFoundError:
        print(f"La jornada {numero} no tiene resultados registrados.")
        return []
    else:
        partidos = []
        for linea in contenido.splitlines():
            if linea.strip():
                p = parsear_partido_seguro(linea)
                if p:
                    partidos.append(p)
        return partidos

# Jornada que sí existe
print("Jornada 1:")
partidos = leer_jornada(1)
for p in partidos:
    print(f"  {p['local']} {p['goles_local']}-{p['goles_visitante']} {p['visitante']}")

# Jornada que NO existe
print("\nJornada 99:")
leer_jornada(99)

Jornada 1:
  Colombia 2-1 Argentina
  Brasil 0-0 Uruguay
  Ecuador 3-2 Peru
  Chile 1-4 Venezuela
  Bolivia 0-2 Paraguay

Jornada 99:
La jornada 99 no tiene resultados registrados.


[]

---

## Sección 3 — JSON: estadísticas que no se pierden

### El problema

Procesamos los partidos, calculamos estadísticas de equipos
(puntos, goles a favor, goles en contra, etc.) pero cuando el programa
termina... **todo se pierde**.

Necesitamos persistir esas estadísticas entre ejecuciones.
JSON es la solución perfecta para esto.

### ¿Qué es JSON en Python?

El flujo siempre es el mismo:

```
[Python] → json.dumps() → [string JSON] → write a archivo
[archivo] → read → [string JSON] → json.loads() → [Python]
```

O en la forma más directa (la que usamos en código real):

```
[Python] → json.dump(f) → [archivo]
[archivo] → json.load(f) → [Python]
```


In [34]:
import json
from pathlib import Path

tabla = {
    "Colombia":  {"pts": 3, "gf": 2, "gc": 1, "pj": 1},
    "Argentina": {"pts": 0, "gf": 1, "gc": 2, "pj": 1},
    "Brasil":    {"pts": 1, "gf": 0, "gc": 0, "pj": 1},
    "Uruguay":   {"pts": 1, "gf": 0, "gc": 0, "pj": 1},
    "Ecuador":   {"pts": 3, "gf": 3, "gc": 2, "pj": 1},
    "Peru":      {"pts": 0, "gf": 2, "gc": 3, "pj": 1},
}

path_tabla = Path("tabla.json")
with open(path_tabla, "w", encoding="utf-8") as f:
    json.dump(tabla, f, indent=4, ensure_ascii=False)

print("Tabla guardada en tabla.json:")
print(path_tabla.read_text(encoding="utf-8"))

Tabla guardada en tabla.json:
{
    "Colombia": {
        "pts": 3,
        "gf": 2,
        "gc": 1,
        "pj": 1
    },
    "Argentina": {
        "pts": 0,
        "gf": 1,
        "gc": 2,
        "pj": 1
    },
    "Brasil": {
        "pts": 1,
        "gf": 0,
        "gc": 0,
        "pj": 1
    },
    "Uruguay": {
        "pts": 1,
        "gf": 0,
        "gc": 0,
        "pj": 1
    },
    "Ecuador": {
        "pts": 3,
        "gf": 3,
        "gc": 2,
        "pj": 1
    },
    "Peru": {
        "pts": 0,
        "gf": 2,
        "gc": 3,
        "pj": 1
    }
}


### Los tres parámetros que importan en `json.dump`

| Parámetro | Qué hace | Sin él |
|-----------|----------|--------|
| `indent=4` | Formato legible con sangría | Todo en una línea ilegible |
| `ensure_ascii=False` | Guarda `é`, `ñ`, `á` tal cual | Los convierte a `\u00e9`, `\u00f1`... |
| `encoding="utf-8"` | En el `open()`, no en `dump` | Falla en Windows con acentos |

> 🇨🇴 `ensure_ascii=False` es **imprescindible** si trabajas con nombres en español.


In [35]:
# Veamos la diferencia con ensure_ascii

equipo = {"nombre": "Atlético Bucaramanga", "ciudad": "Bucaramangá"}

print("Con ensure_ascii=True (por defecto):")
print(json.dumps(equipo, indent=2, ensure_ascii=True))

print()
print("Con ensure_ascii=False:")
print(json.dumps(equipo, indent=2, ensure_ascii=False))

Con ensure_ascii=True (por defecto):
{
  "nombre": "Atl\u00e9tico Bucaramanga",
  "ciudad": "Bucaramang\u00e1"
}

Con ensure_ascii=False:
{
  "nombre": "Atlético Bucaramanga",
  "ciudad": "Bucaramangá"
}


In [36]:
# Cargar la tabla — forma profesional
with open("tabla.json", "r", encoding="utf-8") as f:
    tabla_cargada = json.load(f)

print("Tipo de dato cargado:", type(tabla_cargada))
print()
print(f"{'EQUIPO':<15} {'PJ':>4} {'GF':>4} {'GC':>4} {'PTS':>4}")
print("─" * 35)

for equipo, stats in tabla_cargada.items():
    print(f"{equipo:<15} {stats['pj']:>4} {stats['gf']:>4} {stats['gc']:>4} {stats['pts']:>4}")

Tipo de dato cargado: <class 'dict'>

EQUIPO            PJ   GF   GC  PTS
───────────────────────────────────
Colombia           1    2    1    3
Argentina          1    1    2    0
Brasil             1    0    0    1
Uruguay            1    0    0    1
Ecuador            1    3    2    3
Peru               1    2    3    0


### `dump`/`load` vs `dumps`/`loads` — cuál usar cuándo

| Función | Trabaja con | Cuándo usarla |
|---------|------------|---------------|
| `json.dump(obj, f)` | archivo abierto | guardar a disco |
| `json.load(f)` | archivo abierto | leer desde disco |
| `json.dumps(obj)` | devuelve string | APIs, logs, tests |
| `json.loads(s)` | recibe string | respuestas de APIs |
